<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB12_Training_Deep_Networks_Properly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NB12 · Class 12 — Training Deep Networks Properly**

## Block 3: AI — Deep Learning (continued)

`NB11` built a neural network that *ran* — it trained, it predicted, it landed somewhere near `NB08`'s classical models. This class turns that into a network you'd actually *trust*: a proper train/validation/test split for monitoring training, reading train-vs-validation curves to catch overfitting as it happens, two regularization techniques (dropout, early stopping) to fight it, and a look at how the choice of optimizer changes training itself. Same real Sonar dataset as `NB08`/`NB11`, so every comparison stays direct.

### Learning objectives

By the end of this class, students will be able to:
- Explain why neural networks need a validation set *during* training, not just a test set at the end.
- Read a train-vs-validation loss curve to diagnose overfitting as it happens.
- Explain and apply dropout as a regularization technique, and understand why it only acts during training.
- Implement early stopping to halt training at the right point automatically.
- Compare optimizers (SGD, Adam, RMSprop) and explain what each does differently.
- Combine these techniques into one final, properly trained and honestly evaluated model.

### Agenda (2-hour class)

| # | Class segment | Approx. duration | Type |
|---|---------------------|:---:|:---:|
| 1 | Recap of Block 2–3 so far, today's roadmap | 5 min | Theory |
| 2 | Why neural networks need a validation set during training | 10 min | Theory |
| 3 | Hands-on: train/validation split, tracking both loss curves | 15 min | Practice |
| 4 | Diagnosing overfitting from the curves | 15 min | Theory + Practice |
| 5 | Dropout: theory, structure, and a hands-on comparison | 20 min | Theory + Practice |
| 6 | Early stopping: theory and a hands-on implementation | 20 min | Theory + Practice |
| 7 | Optimizers: SGD, Adam, RMSprop compared | 20 min | Theory + Practice |
| 8 | Putting it together: a final, properly trained model | 25 min | Practice |
| 9 | When is a neural network the right choice? | 5 min | Theory |
| 10 | Summary, homework, next class | 5 min | Theory |

> Timings are approximate guidance, not a strict script — there are no scheduled breaks. If we cover everything with time to spare, class ends early; that can happen and is fine.


---

## 1. Recap: where we are

- **Block 2** (`NB02`–`NB10`): the classical ML toolkit, closing with a full tuned project.
- **`NB11`**: perceptron → MLP theory, a first real PyTorch classifier on the Sonar dataset, evaluated once against `NB08`'s classical models.
- **`NB12`** (today): the training *process* itself — validation monitoring, regularization, optimizers — turning a network that merely runs into one that generalizes.

---

## 2. Why neural networks need a validation set during training

`NB11` tracked only the **training loss**. That tells us the network is fitting *something* — but not whether that something is the real pattern in the data, or just noise in the 166 training examples it saw. This is exactly the overfitting risk from `NB07`/`NB08`, now applied to a model with thousands of trainable weights instead of a single `max_depth` knob.

Recall the three-way split from `NB07`:
- **Training set**: what the network's weights are fit to.
- **Validation set**: monitored *throughout* training, without ever updating weights from it — our early-warning system for overfitting.
- **Test set**: touched exactly once, at the very end, exactly as in `NB10`.

For a neural network, "monitoring the validation set" means computing its loss after every epoch (without calling `loss.backward()` on it) and `watching how the two curves — training and validation — diverge or stay together`.

---

## 3. Hands-on: train/validation split, tracking both loss curves

Reload the same real Sonar dataset as `NB08`/`NB11`, but this time split it three ways: 60% train, 20% validation, 20% test.

In [ ]:
!wget -q -O sonar.csv https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/sonar.all-data

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sonar = pd.read_csv("sonar.csv", header=None)
sonar.columns = [f"freq_{i}" for i in range(60)] + ["label"]

X = sonar.drop(columns="label").values
y = (sonar["label"] == "M").astype(int).values

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full
)  # 0.25 of the remaining 80% = 20% of the total

print("Train:", X_train.shape, " Val:", X_val.shape, " Test:", X_test.shape)

Scale using statistics from the **training set only** — fitting the scaler on validation or test data would be the same leakage mistake flagged back in `NB07`/`NB08`. Then convert everything to tensors:

In [ ]:
import torch
import torch.nn as nn

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

Reuse `NB11`'s architecture, then train for many epochs — deliberately more than we probably need — while recording *both* losses every epoch:

In [ ]:
class SonarMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_features, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.layers(x)

torch.manual_seed(42)
model = SonarMLP(n_features=X_train_t.shape[1])
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

n_epochs = 400
train_losses, val_losses = [], []

for epoch in range(n_epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_val_t), y_val_t)
    val_losses.append(val_loss.item())

Note the `model.train()` / `model.eval()` calls — a habit worth building now, since it matters even more once dropout enters the picture in Part 5. Plot both curves together:

In [ ]:
import matplotlib.pyplot as plt

plt.plot(train_losses, label="Training loss")
plt.plot(val_losses, label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss (binary cross-entropy)")
plt.title("Training vs. validation loss, 400 epochs, no regularization")
plt.legend()
plt.show()

---

## 4. Diagnosing overfitting from the curves

**Read your own plot** — this is the neural-network equivalent of `NB08`'s decision-tree depth-vs-accuracy demo:
- If both curves fall together and roughly track each other, the network is generalizing — keep going.
- If the training loss keeps falling while the validation loss flattens out or starts *rising*, that gap **is** overfitting, made visible exactly as it was for the decision tree — except here the "knob" isn't depth, it's simply how many epochs we let it train, combined with how much capacity (layers × units) the network has relative to how much data (208 examples total) it has to learn from.

With a small, high-capacity network on only 166 training examples, `some degree of this gap by epoch 400 would not be surprising`. The next two sections cover the two most common fixes.

**Try it yourself**: turn the visual read into two real numbers — the final train/validation gap, and the exact epoch where validation loss reached its best (lowest) value.

In [ ]:
import numpy as np

final_gap = val_losses[-1] - train_losses[-1]
best_val_epoch = int(np.argmin(val_losses))

print(f"Final train loss: {train_losses[-1]:.4f}, final val loss: {val_losses[-1]:.4f}, gap: {final_gap:.4f}")
print(f"Validation loss reached its minimum at epoch {best_val_epoch + 1} (of {n_epochs})")


---

## 5. Dropout

**[Dropout](https://en.wikipedia.org/wiki/Dropout_%28neural_networks%29)** randomly deactivates a fraction of neurons — different ones each forward pass — *only during training*. Each pass, `the network is forced to make good predictions without relying on any single neuron always being present`, which discourages neurons from co-adapting to noise specific to the training set. At evaluation time, dropout is switched off and every neuron participates (which is exactly why `model.eval()` matters — it's what tells PyTorch's `Dropout` layers to stop dropping).

Let's visualize what one training step with dropout looks like, compared to a normal (fully connected) pass:

In [ ]:
import matplotlib.patches as patches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

input_y = np.linspace(0, 4, 4)
hidden_y = np.linspace(0, 4, 5)
output_y = np.linspace(1, 3, 2)
dropped_hidden = {1, 3}

for ax, title, use_dropout in zip(
    axes, ["Without dropout", "With dropout (this training step)"], [False, True]
):
    for iy in input_y:
        for j, hy in enumerate(hidden_y):
            dropped = use_dropout and j in dropped_hidden
            ax.plot([0, 2], [iy, hy], color="lightgray" if dropped else "steelblue",
                     lw=0.7 if dropped else 1.2, zorder=1)

    for j, hy in enumerate(hidden_y):
        dropped = use_dropout and j in dropped_hidden
        for oy in output_y:
            ax.plot([2, 4], [hy, oy], color="lightgray" if dropped else "steelblue",
                     lw=0.7 if dropped else 1.2, zorder=1)

    for iy in input_y:
        ax.add_patch(patches.Circle((0, iy), 0.25, facecolor="lightblue", edgecolor="black", zorder=2))
    for j, hy in enumerate(hidden_y):
        dropped = use_dropout and j in dropped_hidden
        ax.add_patch(patches.Circle((2, hy), 0.25, facecolor="lightgray" if dropped else "lightcoral",
                                     edgecolor="black", zorder=2))
        if dropped:
            ax.text(2, hy, "x", ha="center", va="center", fontsize=12, color="dimgray", zorder=3)
    for oy in output_y:
        ax.add_patch(patches.Circle((4, oy), 0.25, facecolor="lightgreen", edgecolor="black", zorder=2))

    ax.set_xlim(-1, 5)
    ax.set_ylim(-1, 5)
    ax.axis("off")
    ax.set_title(title)

plt.tight_layout()
plt.show()

Every forward pass during training drops a *different* random subset — the diagram shows one example step, not a fixed pattern. Now add `nn.Dropout` to our architecture and retrain, tracking both curves exactly as in Part 3:

In [ ]:
class SonarMLPDropout(nn.Module):
    def __init__(self, n_features, dropout_rate=0.3):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_features, 32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.layers(x)

torch.manual_seed(42)
dropout_model = SonarMLPDropout(n_features=X_train_t.shape[1], dropout_rate=0.3)
optimizer = torch.optim.Adam(dropout_model.parameters(), lr=0.001)

train_losses_do, val_losses_do = [], []
for epoch in range(n_epochs):
    dropout_model.train()
    optimizer.zero_grad()
    loss = criterion(dropout_model(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()
    train_losses_do.append(loss.item())

    dropout_model.eval()
    with torch.no_grad():
        val_losses_do.append(criterion(dropout_model(X_val_t), y_val_t).item())

plt.plot(train_losses_do, label="Training loss (with dropout)")
plt.plot(val_losses_do, label="Validation loss (with dropout)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs. validation loss, with dropout")
plt.legend()
plt.show()

**Compare this plot to Part 4's**: does the gap between training and validation loss open up more slowly, or stay smaller overall? Dropout usually makes training loss fall *more slowly* (the network can't fit the training data as eagerly) in exchange for a validation loss that holds up better — a direct, visible trade of a little training performance for better generalization.

> **Further reading**: [Dropout (Wikipedia)](https://en.wikipedia.org/wiki/Dropout_%28neural_networks%29) · [Regularization (Wikipedia)](https://en.wikipedia.org/wiki/Regularization_%28mathematics%29) · [`torch.nn.Dropout` documentation](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html).

---

## 6. Early stopping

Dropout changes *what* the network can fit; **[early stopping](https://en.wikipedia.org/wiki/Early_stopping)** changes *how long* we let it try. The idea: keep a copy of the model's weights whenever validation loss reaches a new best, and if it fails to improve for a set number of epochs (the **patience**), stop training and restore that best copy — `instead of blindly training for a fixed number of epochs regardless of what the validation curve is doing`.

In [ ]:
import copy

torch.manual_seed(42)
es_model = SonarMLPDropout(n_features=X_train_t.shape[1], dropout_rate=0.3)
optimizer = torch.optim.Adam(es_model.parameters(), lr=0.001)

patience = 25
best_val_loss = float("inf")
patience_counter = 0
best_state = None
max_epochs = 400

for epoch in range(max_epochs):
    es_model.train()
    optimizer.zero_grad()
    loss = criterion(es_model(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()

    es_model.eval()
    with torch.no_grad():
        val_loss = criterion(es_model(X_val_t), y_val_t).item()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(es_model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch + 1} (best validation loss: {best_val_loss:.4f})")
            break

es_model.load_state_dict(best_state)

**Try it yourself**: compare the epoch at which early stopping halted to Section 4's divergence point (`best_val_epoch`, computed above) — they use different models (this one has dropout) so won't match exactly, but should be in a broadly similar range if both are responding to the same underlying overfitting signal.

In [ ]:
print(f"Early stopping halted at epoch {epoch + 1}; the best validation loss was seen roughly "
      f"{patience} epochs earlier, around epoch {epoch + 1 - patience}.")
print(f"Section 4's val-loss-minimum epoch (no dropout): {best_val_epoch + 1}")


**Read your own output**: did training stop before reaching `max_epochs`? If so, at roughly what epoch — does that line up with where Part 3's original curves started to diverge? `best_state` now holds the weights from the *best* validation epoch, not necessarily the last one — exactly the model we actually want to keep.

> **Further reading**: [Early stopping (Wikipedia)](https://en.wikipedia.org/wiki/Early_stopping).

---

## 7. Optimizers: SGD, Adam, RMSprop compared

`NB11` used **Adam** without much explanation. All three common optimizers implement the same core idea from `NB11` Part 5 (nudge weights against the gradient), but differ in *how*:

| Optimizer | Core idea | Typical behavior |
|---|---|---|
| **[SGD](https://en.wikipedia.org/wiki/Stochastic_gradient_descent)** (with momentum) | Plain gradient steps, optionally accumulating a "velocity" from past steps | Simple, well understood, often needs more careful learning-rate tuning |
| **Adam** | Adapts the learning rate *per weight*, using running estimates of recent gradients | Fast convergence, good default for many problems, used throughout `NB11`/`NB12` so far |
| **RMSprop** | Also adapts the learning rate per weight, using a running average of squared gradients | A predecessor/relative of Adam, often competitive on recurrent-style problems |

Let's compare their training-loss curves directly, on the same architecture and data, with everything else held fixed:

In [ ]:
def train_and_track(optimizer_name, n_epochs=150):
    torch.manual_seed(42)
    m = SonarMLP(n_features=X_train_t.shape[1])
    if optimizer_name == "SGD":
        opt = torch.optim.SGD(m.parameters(), lr=0.01, momentum=0.9)
    elif optimizer_name == "Adam":
        opt = torch.optim.Adam(m.parameters(), lr=0.001)
    else:
        opt = torch.optim.RMSprop(m.parameters(), lr=0.001)

    losses = []
    for _ in range(n_epochs):
        m.train()
        opt.zero_grad()
        loss = criterion(m(X_train_t), y_train_t)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses

optimizer_curves = {name: train_and_track(name) for name in ["SGD", "Adam", "RMSprop"]}

for name, losses in optimizer_curves.items():
    plt.plot(losses, label=name)
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("Optimizer comparison (same architecture, same data)")
plt.legend()
plt.show()

**Try it yourself**: the plot only shows *training* loss — extend the comparison to each optimizer's final *validation* loss too, so "fastest training convergence" and "best generalization" can be compared directly, not assumed to be the same thing.

In [ ]:
def train_and_track_with_val(optimizer_name, n_epochs=150):
    torch.manual_seed(42)
    m = SonarMLP(n_features=X_train_t.shape[1])
    if optimizer_name == "SGD":
        opt = torch.optim.SGD(m.parameters(), lr=0.01, momentum=0.9)
    elif optimizer_name == "Adam":
        opt = torch.optim.Adam(m.parameters(), lr=0.001)
    else:
        opt = torch.optim.RMSprop(m.parameters(), lr=0.001)

    for _ in range(n_epochs):
        m.train()
        opt.zero_grad()
        loss = criterion(m(X_train_t), y_train_t)
        loss.backward()
        opt.step()

    m.eval()
    with torch.no_grad():
        final_val_loss = criterion(m(X_val_t), y_val_t).item()
    return final_val_loss

for name in ["SGD", "Adam", "RMSprop"]:
    print(f"{name}: final validation loss = {train_and_track_with_val(name):.4f}")


**Interpret your own plot**: which optimizer reaches a low loss fastest? Does SGD lag behind the two adaptive methods, as the table above would predict? Try changing SGD's learning rate (`lr=0.01` → `lr=0.1` or `lr=0.001`) and re-running — how sensitive is plain SGD to that choice, compared to Adam/RMSprop?

> **Further reading**: [Stochastic gradient descent (Wikipedia)](https://en.wikipedia.org/wiki/Stochastic_gradient_descent) · [`torch.optim.SGD` documentation](https://pytorch.org/docs/stable/generated/torch.optim.SGD.html) · [`torch.optim.RMSprop` documentation](https://pytorch.org/docs/stable/generated/torch.optim.RMSprop.html).

---

## 8. Putting it together: a final, properly trained model

Combine everything: dropout, early stopping, Adam — then, and only then, evaluate on the test set we haven't touched since Part 3.

In [ ]:
torch.manual_seed(42)
final_model = SonarMLPDropout(n_features=X_train_t.shape[1], dropout_rate=0.3)
optimizer = torch.optim.Adam(final_model.parameters(), lr=0.001)

best_val_loss = float("inf")
patience_counter = 0
best_state = None

for epoch in range(max_epochs):
    final_model.train()
    optimizer.zero_grad()
    loss = criterion(final_model(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()

    final_model.eval()
    with torch.no_grad():
        val_loss = criterion(final_model(X_val_t), y_val_t).item()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(final_model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Stopped at epoch {epoch + 1}")
            break

final_model.load_state_dict(best_state)

Evaluate the final model on the test set we haven't touched since Part 3:

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

final_model.eval()
with torch.no_grad():
    test_probs = torch.sigmoid(final_model(X_test_t))
    test_preds = (test_probs > 0.5).float()

y_pred_np = test_preds.numpy().ravel()
y_test_np = y_test_t.numpy().ravel()

print(confusion_matrix(y_test_np, y_pred_np))
print()
print(classification_report(y_test_np, y_pred_np, target_names=["Rock", "Mine"]))

**Try it yourself**: make the comparison in the reflection below concrete. Evaluate Section 3's original, unregularized model on this same test set (it was never touched by the test set before) and compare its accuracy directly to `final_model`'s.

In [ ]:
model.eval()
with torch.no_grad():
    unreg_test_preds = (torch.sigmoid(model(X_test_t)) > 0.5).float()
unreg_test_accuracy = (unreg_test_preds == y_test_t).float().mean().item()
final_test_accuracy = (test_preds == y_test_t).float().mean().item()

print(f"Section 3 model (no regularization) test accuracy: {unreg_test_accuracy:.3f}")
print(f"Section 8 model (dropout + early stopping)  test accuracy: {final_test_accuracy:.3f}")


See which one wins on your run. With a test set of only ~40 rows, a difference of one or two predictions can flip which model looks better — if regularization doesn't clearly win here, that's an honest, real result, not a failure of the method; Section 9's point about small tabular datasets applies just as much within this notebook as it does against `NB08`'s classical models.

**The three-way comparison that matters**: how does this test accuracy compare to (a) `NB11`'s untuned MLP, and (b) `NB08`'s best classical model? A properly regularized network *should* hold up at least as well as `NB11`'s version, and ideally close some of the gap to the tuned classical models from `NB08` — though on a dataset this small (208 rows total), `classical tree-based methods often remain very competitive with neural networks, which tend to need more data to show a clear advantage`.

---

## 9. When is a neural network the right choice?

After two classes of real comparison against `NB08`'s classical models on the *same* tabular dataset, a fair conclusion: for small, tabular, already-feature-engineered problems like ours, `tree-based ensembles are often just as good, faster to train, and far easier to tune`. Neural networks earn their extra complexity mainly when:

| Situation | Why NNs help |
|---|---|
| Raw, unstructured data (images, audio, long sequences) | Automatic feature learning replaces hand-engineering — the motivation from `NB11` Part 2 |
| Very large datasets | Deep networks generally keep improving with more data; classical models plateau sooner |
| Transfer learning is available | A network pretrained on a huge dataset can be adapted to a small one — `NB13`'s CNN class will use exactly this |

This sets up `NB13` precisely: our next real dataset (underwater imagery) is exactly the "raw, unstructured data" case where a CNN's automatic feature learning genuinely outperforms hand-engineered tabular features.

---

## Class summary

- Neural networks need a validation set monitored *during* training, not just a final test set — tracked every epoch, not just once.
- A training loss that keeps falling while validation loss flattens or rises is overfitting, visible directly in the two curves.
- Dropout randomly deactivates neurons during training only, discouraging over-reliance on any single one; `model.train()`/`model.eval()` control whether it's active.
- Early stopping keeps the best-validation-loss weights and halts training automatically, instead of guessing a fixed epoch count.
- SGD, Adam, and RMSprop all implement gradient descent differently — Adam/RMSprop adapt their learning rate per weight and usually converge faster.
- On small tabular datasets, a properly regularized neural network is competitive with — but not automatically better than — a tuned classical model; the real advantage shows up on raw, unstructured, or much larger data.

## For the next class (NB13)

We move to **Convolutional Neural Networks (CNNs)**: the architecture built specifically for image data, applied to real underwater inspection imagery — exactly the kind of raw, unstructured data where Part 9's argument for deep learning actually pays off.

## Homework / Practice Ideas

1. Change `dropout_rate` to `0.1` and to `0.5` in Part 5 — how does each change the gap between training and validation loss compared to `0.3`?
2. Change `patience` in Part 6 to `5` and to `50` — how does the epoch at which training stops change, and how does final test accuracy change?
3. Add a fourth optimizer to Part 7's comparison, `torch.optim.Adagrad` — how does its curve compare to the other three?
4. Combine dropout with a *larger* network (e.g., hidden layers of 128 and 64 instead of 32 and 16) — does dropout let the bigger network avoid overfitting as effectively as the smaller one did?
5. In your own words, explain why we compute validation loss inside a `with torch.no_grad():` block — what would go wrong (or just be wasted effort) if we didn't?

> ***As always: a lower training loss is not an achievement by itself — it only matters if validation and test performance improve along with it.***
